> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **GPT Judge Evaluation**
This notebook measures *Confirmation Bias* using LLM-as-a-judge (with gpt-4o), instructing it to rate the coherence on a scale of 0-10.

In [1]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.gpt_judge import compute_gpt_metrics

/Users/fabrizio/Desktop/Tesi/confirmation-bias-analysis/src/config.py:17: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [2]:
# Define the path to the generated results (JSONL format)
# file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
file_path = "../data/interim/5_mmlu_pro_gpt_4o_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

Loaded 500 samples.


## **Evaluation**
Compute the evaluation metrics by sending the generated results to GPT, and extract the final score (parsed list).

In [3]:
# Compute the GPT metrics
df_evaluated = compute_gpt_metrics(df_results, model="gpt-4o", sleep_time=1.0)

Starting GPT evaluation as judge for 500 samples...
Row 1/500 evaluated.
Row 2/500 evaluated.
Row 3/500 evaluated.
Row 4/500 evaluated.
Row 5/500 evaluated.
Row 6/500 evaluated.
Row 7/500 evaluated.
Row 8/500 evaluated.
Row 9/500 evaluated.
Row 10/500 evaluated.
Row 11/500 evaluated.
Row 12/500 evaluated.
Row 13/500 evaluated.
Row 14/500 evaluated.
Row 15/500 evaluated.
Row 16/500 evaluated.
Row 17/500 evaluated.
Row 18/500 evaluated.
Row 19/500 evaluated.
Row 20/500 evaluated.
Row 21/500 evaluated.
Row 22/500 evaluated.
Row 23/500 evaluated.
Row 24/500 evaluated.
Row 25/500 evaluated.
Row 26/500 evaluated.
Row 27/500 evaluated.
Row 28/500 evaluated.
Row 29/500 evaluated.
Row 30/500 evaluated.
Row 31/500 evaluated.
Row 32/500 evaluated.
Row 33/500 evaluated.
Row 34/500 evaluated.
Row 35/500 evaluated.
Row 36/500 evaluated.
Row 37/500 evaluated.
Row 38/500 evaluated.
Row 39/500 evaluated.
Row 40/500 evaluated.
Row 41/500 evaluated.
Row 42/500 evaluated.
Row 43/500 evaluated.
Row 44/500 

## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [4]:
# Aggregation of the average scores for each prompt type
mean_neutral = df_evaluated["score_neutral"].mean()
mean_leading = df_evaluated["score_leading"].mean()
mean_contra = df_evaluated["score_contradictory"].mean()

print("Results of the GPT Judge Evaluation:")
print(f"Average Support (Neutral Prompt):        {mean_neutral:.2f} / 10")
print(f"Average Support (Leading Prompt):        {mean_leading:.2f} / 10")
print(f"Average Support (Contradictory Prompt):  {mean_contra:.2f} / 10")

# If the Leading prompt increases adherence and the Contradictory prompt decreases it significantly, 
# the numerical discrepancy indicates that the tested model suffers from confirmation bias.

display(df_evaluated[["sample", "model", "claim", "score_neutral", "score_leading", "score_contradictory"]].head(10))

Results of the GPT Judge Evaluation:
Average Support (Neutral Prompt):        8.82 / 10
Average Support (Leading Prompt):        7.10 / 10
Average Support (Contradictory Prompt):  5.97 / 10


,sample,model,claim,score_neutral,score_leading,score_contradictory
0,1,gpt-4o,The classic aging pattern is defined as 2 decl...,8.5,8.5,8.5
1,2,gpt-4o,A listener-sponsored radio station wants to es...,9.5,8.5,0.0
2,3,gpt-4o,A wife and husband are married and own a dairy...,10.0,10.0,10.0
3,4,gpt-4o,The Yang-shao culture gave way to the Lung-Sha...,9.5,9.0,9.0
4,5,gpt-4o,"Explain how a ""Skinner box"" can be used in ope...",10.0,10.0,9.0
5,6,gpt-4o,"The term ""senescence"" refers to",9.5,7.5,5.0
6,7,gpt-4o,"In the extreme case, $ 1,000 of new reserves p...",9.0,8.5,9.0
7,8,gpt-4o,A prisoner was serving a life sentence in a st...,10.0,7.5,10.0
8,9,gpt-4o,It is correct to state that for the treatment ...,10.0,8.5,10.0
9,10,gpt-4o,A surgical support can be used for:,8.5,9.5,7.0


## **Exporting**
Export the results to a CSV file for later use in the final analysis notebook.

In [5]:
# Extract filename from the input path
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
output_file = f"../data/interim/{base_name}_gpt.csv"

# Save the evaluated DataFrame to the interim data directory
df_evaluated.to_csv(output_file, index=False)
print(f"Saved evaluation file to {output_file}")

Saved evaluation file to ../data/interim/5_mmlu_pro_gpt_4o_gpt.csv
